# 01 · Exploratory data analysis

LoanShield classifies Indian lending apps on Google Play as **verified** (listed in RBI's Digital Lending Apps directory) or **unverified**. This notebook describes the collected data, the labels and how the two classes differ, using only files under `data/` (no network).

Run after Phases 1–3 (`collect`, `label.build_labels`, `features.build_features`). Figures are saved to `results/figures/eda_*.png`.

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, HTML, display, Markdown
from loanshield.common import ROOT, path, cfg
os.chdir(ROOT)                      # all paths are relative to the repo root
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_colwidth", 60, "display.max_columns", 30)
print("repo:", ROOT)

from loanshield.models.evaluate import SERIES, SURFACE, INK, MUTED, GRID
VER, UNV = SERIES[0], SERIES[1]     # fixed categorical order: verified = blue, unverified = orange

def style(ax, title):
    ax.set_facecolor(SURFACE); ax.set_title(title, loc="left", color=INK, fontsize=11)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]: ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0); ax.grid(axis="x", color=GRID); ax.set_axisbelow(True)

def save(fig, name):
    out = path("results", "figures", name)
    fig.tight_layout(); fig.savefig(out, dpi=150, facecolor=SURFACE); plt.close(fig)
    display(Image(str(out)))

## 1. Collection funnel
Candidates come from Play search (55 terms), a one-hop "similar apps" snowball, and the Play links in the RBI list. Only Finance apps that mention lending are kept.

In [ ]:
cands = pd.read_csv(path("interim", "candidates.csv"))
flog = pd.read_csv(path("interim", "filter_log.csv"))
apps = pd.read_csv(path("interim", "apps.csv"))
rbi = pd.read_csv(sorted(path("raw").glob("rbi_dla_2*.csv"))[-1])
print(f"RBI DLA rows: {len(rbi)} | unique REs: {rbi.re_name.nunique()} | Play package IDs: {rbi.package_id.nunique()}")
display(cands["source"].value_counts().rename("candidates by discovery source").to_frame())
display(flog["reason"].str.replace(r"genre:.*", "genre: not Finance", regex=True).value_counts().rename("filter decision").to_frame())
print("kept apps:", len(apps))

## 2. Labels
Automatic decisions (RBI package / name matching, offers-credit rule, seed list) plus the manually reviewed queue. `excluded` apps do not lend and are not modelled.

In [ ]:
dec = pd.read_csv(path("interim", "label_decisions.csv"))
labels = pd.read_csv(path("processed", "labels.csv"))
src = cands.set_index("app_id")["source"]
display(dec["auto_label"].value_counts().rename("automatic decision").to_frame())
display(pd.crosstab(labels["label"], labels["label_source"], margins=True))
print("Discovery bias check: decision x discovery source")
display(pd.crosstab(dec["auto_label"], dec["app_id"].map(src), margins=True))

**Discovery bias.** Every app found through the RBI list's Play links is verified by construction. A model could learn "looks like an app found via the RBI list" instead of risk, so the headline results in `02_models` are on *organic* apps (search / similar only). This is a team decision recorded in `config.yaml`.

## 3. How the classes differ
Features from `data/processed/features.parquet` joined to the labels.

In [ ]:
X = pd.read_parquet(path("processed", "features.parquet")).merge(labels[["app_id", "label"]], on="app_id")
groups = json.loads(path("processed", "feature_groups.json").read_text(encoding="utf-8"))
n = X.label.value_counts()
print(n.to_dict())

binary = [c for c in groups if X[c].dropna().isin([0, 1]).all() and X[c].nunique() > 1]
prev = X.groupby("label")[binary].mean().T
prev["gap"] = prev["unverified"] - prev["verified"]
prev = prev.reindex(prev["gap"].abs().sort_values(ascending=False).index).head(14)[::-1]

fig, ax = plt.subplots(figsize=(8, 6), facecolor=SURFACE)
y = np.arange(len(prev)); h = 0.38
ax.barh(y + h / 2, prev["verified"], h, color=VER, label=f"verified (n={n.get('verified', 0)})")
ax.barh(y - h / 2, prev["unverified"], h, color=UNV, label=f"unverified (n={n.get('unverified', 0)})")
ax.set_yticks(y, prev.index, color=INK, fontsize=8); ax.set_xlim(0, 1)
ax.set_xlabel("share of apps", color=INK); ax.legend(frameon=False, loc="lower right", fontsize=8)
style(ax, "Binary features with the largest class gap")
save(fig, "eda_binary_feature_gaps.png")

In [ ]:
cont = ["app_age_days", "log_min_installs", "log_ratings", "score", "n_dangerous_perms", "urgency_terms",
        "log_desc_len", "rev_mean_score", "rev_share_harass", "rev_share_fraud"]
summ = X.groupby("label")[cont].median().T
summ.columns = [f"median ({c})" for c in summ.columns]
display(summ.round(3))

fig, axes = plt.subplots(2, 3, figsize=(11, 6), facecolor=SURFACE)
for ax, c in zip(axes.flat, ["app_age_days", "log_min_installs", "n_dangerous_perms",
                             "urgency_terms", "rev_share_harass", "rev_share_fraud"]):
    data = [X.loc[X.label == k, c].dropna() for k in ["verified", "unverified"]]
    bp = ax.boxplot(data, vert=False, widths=0.5, patch_artist=True, showfliers=False,
                    medianprops={"color": INK}, whiskerprops={"color": MUTED}, capprops={"color": MUTED})
    for patch, col in zip(bp["boxes"], [VER, UNV]):
        patch.set_facecolor(col); patch.set_edgecolor(col); patch.set_alpha(0.75)
    ax.set_yticks([1, 2], ["verified", "unverified"], color=INK, fontsize=8)
    style(ax, c)
save(fig, "eda_feature_distributions.png")

## 4. Release dates
Unverified apps are much newer, which matters for the temporal hold-out in `02_models` and for interpreting `app_age_days`.

In [ ]:
X["year"] = pd.to_datetime(labels.set_index("app_id").loc[X.app_id, "first_seen_date"].values, errors="coerce").year
yr = pd.crosstab(X["year"], X["label"]).reindex(columns=["verified", "unverified"], fill_value=0)
display(yr.T)
fig, ax = plt.subplots(figsize=(9, 3.6), facecolor=SURFACE)
ax.bar(yr.index - 0.2, yr["verified"], 0.4, color=VER, label="verified")
ax.bar(yr.index + 0.2, yr["unverified"], 0.4, color=UNV, label="unverified")
ax.set_ylabel("apps", color=INK); ax.legend(frameon=False, fontsize=8)
style(ax, "Apps by release year"); ax.grid(axis="y", color=GRID); ax.grid(axis="x", visible=False)
save(fig, "eda_release_year.png")

## 5. Reviews
Cleaned, de-duplicated review text (`reviews_clean.parquet`); lexicon hits for harassment, hidden fees and fraud (English + Hinglish).

In [ ]:
rc = pd.read_parquet(path("processed", "reviews_clean.parquet"))
rc = rc.merge(labels[["app_id", "label"]], on="app_id")
print(f"{len(rc):,} unique reviews from {rc.app_id.nunique()} labelled apps")
lex = rc.groupby("label")[["lex_harass", "lex_fee", "lex_fraud"]].mean().T
lex.index = ["harassment / threats", "hidden fees", "fraud / scam"]
display((100 * lex).round(2).rename(columns=lambda c: f"% of reviews ({c})"))
display(rc[rc.lex_harass == 1].sample(min(5, int(rc.lex_harass.sum())), random_state=42)[["label", "score", "text"]])

**Observations to carry into the report** (re-read once labels are final):
- How large the unverified class is, and how many come from each label source.
- Which listing-text and permission gaps are largest. Note that RBI-listed lenders also request SMS access for credit checks.
- Whether review lexicons separate the classes, or mostly track app size.